In [6]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML 


dt=1e-6 ; m1=1; g=9.8 ; a=1 ; b=3; w=0.5;
tf=15; N=int(tf/dt);
theta0 = np.pi/4              #condiciones iniciales

In [7]:
#Funciones

def Cart(a,b,w,theta,t):
  xm = a * np.cos(w*t) + b * np.sin(theta)
  ym = a * np.sin(w*t) - b * np.cos(theta)
  return xm, ym


def Ec_EuLag(a,b,w,g,theta,t):
    dwdt=-(1/b) * ( g*np.sin(theta) - a*w*w*np.cos(theta - w*t) ) 
    return dwdt

def Motion(a,b,w,g,theta,wm,am,t):
    for i in range(len(t)-1):
        dt = t[i+1] - t[i]
        # predictores de aceleración, velocidad y posición (Euler)
        am_pred = Ec_EuLag(a,b,w,g,theta[i],t[i])  # aceleración

        wm_pred = wm[i] + am_pred * dt  # Euler vx(i+1)=vx(i)+a_pred*dt ;
        theta_pred = theta[i] + wm[i] * dt  # Euler x(i+1)=x(i)+vx_pred*dt ;

        # correctores de posición, aceleración y velocidad (HEUN)
        theta[i+1] = theta[i] + 0.5 * (wm[i] + wm_pred) * dt
        am_next = Ec_EuLag(a,b,w,g,theta[i+1],t[i+1])
        wm[i+1] = wm[i] + 0.5 * (am_pred + am_next) * dt

    return theta, wm, am

In [8]:
#Inicialización
t=np.linspace(0,tf,N)

rm = np.zeros((2, N))
theta, wm, am = (np.zeros(N) for _ in range(3))
theta[0]=theta0

am[0] = Ec_EuLag(a,b,w,g,theta[0],t[0])


In [9]:
#Principal loop
theta, wm, am =Motion(a,b,w,g,theta,wm,am,t)
rm[0,:] , rm[1,:] = Cart(a,b,w,theta,t)


In [ ]:
#Animación
T_anim  = tf       # segundos simulados a animar (máx. tf)
fps     = 30       # fotogramas por segundo
speed   = 1.0      # 1 = tiempo real, 0.5 = cámara lenta
trail_s = 3.0      # duración de la estela de la masa (s); 0 = sin estela

# Submuestreo: un fotograma cada 'step' pasos de simulación
step    = max(1, round(speed / (fps * dt)))
idx     = np.arange(0, min(N, int(T_anim / dt)), step)
trail_n = int(trail_s / (dt * step))

xm, ym = rm[0, idx], rm[1, idx]                      # masa
xp, yp = a*np.cos(w*t[idx]), a*np.sin(w*t[idx])      # punto de suspensión (sobre el círculo)

# Figura
fig, ax = plt.subplots(figsize=(6, 6))
L = (a + b) * 1.1
ax.set(xlim=(-L, L), ylim=(-L, L), aspect='equal', xlabel='x (m)', ylabel='y (m)',
       title='Péndulo con soporte en movimiento circular')
ax.grid(alpha=0.3)

phi = np.linspace(0, 2*np.pi, 200)
ax.plot(a*np.cos(phi), a*np.sin(phi), 'k-', lw=1)     # círculo de radio a

radius, = ax.plot([], [], 'k--', lw=1)                # radio: centro -> punto de suspensión
rod,    = ax.plot([], [], 'k-', lw=2)
pivot,  = ax.plot([], [], 'ko', ms=6)
bob,    = ax.plot([], [], 'o', color='tab:red', ms=10)
trail,  = ax.plot([], [], '-', color='tab:red', lw=1, alpha=0.6)
time_txt = ax.text(0.03, 0.95, '', transform=ax.transAxes)

def update(k):
    radius.set_data([0, xp[k]], [0, yp[k]])
    rod.set_data([xp[k], xm[k]], [yp[k], ym[k]])
    pivot.set_data([xp[k]], [yp[k]])
    bob.set_data([xm[k]], [ym[k]])
    if trail_n > 0:
        j = max(0, k - trail_n)
        trail.set_data(xm[j:k+1], ym[j:k+1])
    time_txt.set_text(f't = {t[idx[k]]:.2f} s')
    return radius, rod, pivot, bob, trail, time_txt

anim = FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True) \
    if 'FuncAnimation' in globals() else \
    animation.FuncAnimation(fig, update, frames=len(idx), interval=1000/fps, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())